## Imports and Setup

In [ ]:
import sys

sys.path.append('..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix

from src import (
    # Models
    RewardModelEnsemble,
    NeuralGreedy, NeuralEpsilon, NeuralUCB, NeuralThompson,
    # Policies
    LinUCBPolicy, EpsilonGreedyPolicy, BoltzmannPolicy,
    GreedyPolicy, RandomPolicy,
    # Evaluation
    OfflinePolicyEvaluator, actions_to_probs, softmax_probs,
    counterfactual_policy_value,
    # Simulation
    OnlineSimulator, make_reward_model_agent, make_neural_bandit_agent,
    make_linucb_agent, make_random_agent, make_oracle_agent,
    # Pipeline
    FeaturePipeline,
    # Data
    TREATMENTS, N_TREATMENTS, IDX_TO_TREATMENT, reward_oracle,
    # Utils
    seed_everything, setup_plotting, timer, save_results,
    TREATMENT_COLORS, ensure_dirs,
)
from src.utils import (
    plot_cumulative_regret, plot_learning_curves,
    plot_regret_and_accuracy, AGENT_COLORS,
)

seed_everything(42)
setup_plotting()
ensure_dirs()
print("Setup complete")

## Load Data and Prepare Both Pipelines

In [ ]:
df = pd.read_csv("../data/bandit_dataset.csv")

pipe_u = FeaturePipeline(scale=False, add_interactions=True)
X_train_u, X_test_u, meta_u = pipe_u.fit_transform_split(df, test_size=0.2, seed=42)

pipe_s = FeaturePipeline(scale=True, add_interactions=True)
X_train_s, X_test_s, meta_s = pipe_s.fit_transform_split(df, test_size=0.2, seed=42)

feature_names = meta_u['feature_names']
input_dim = X_train_s.shape[1]
n_test = len(X_test_u)

print(f"Train: {X_train_u.shape[0]}  Test: {n_test}  Features: {input_dim}")

## Train All Models

In [ ]:
models = {}

# ── XGBoost Ensemble ──
print("Training XGBoost Ensemble...")
xgb = RewardModelEnsemble()
xgb.fit(X_train_u, meta_u['a_train'], meta_u['y_train'], feature_names=feature_names)
models['XGBoost'] = xgb

# ── NeuralGreedy ──
print("Training NeuralGreedy...")
ng = NeuralGreedy(input_dim=input_dim, hidden_dims=[128, 64], device="cpu")
ng.train(X_train_s, meta_s['a_train'], meta_s['y_train'], epochs=60, verbose=False)
models['NeuralGreedy'] = ng

# ── NeuralEpsilon ──
print("Training NeuralEpsilon...")
ne = NeuralEpsilon(input_dim=input_dim, hidden_dims=[128, 64], epsilon=0.1, device="cpu")
ne.train(X_train_s, meta_s['a_train'], meta_s['y_train'], epochs=60, verbose=False)
models['NeuralEpsilon'] = ne

# ── NeuralUCB ──
print("Training NeuralUCB...")
nucb = NeuralUCB(input_dim=input_dim, hidden_dims=[128, 64], alpha=1.0, device="cpu")
nucb.train(X_train_s, meta_s['a_train'], meta_s['y_train'], epochs=60, verbose=False)
for i in range(min(5000, len(X_train_s))):
    nucb.update_covariance(X_train_s[i], int(meta_s['a_train'][i]))
models['NeuralUCB'] = nucb

# ── NeuralThompson ──
print("Training NeuralThompson...")
nts = NeuralThompson(input_dim=input_dim, hidden_dims=[128, 64], noise_variance=0.25, device="cpu")
nts.train(X_train_s, meta_s['a_train'], meta_s['y_train'], epochs=60, verbose=False)
for i in range(min(5000, len(X_train_s))):
    nts.update_posterior(X_train_s[i], int(meta_s['a_train'][i]), float(meta_s['y_train'][i]))
models['NeuralThompson'] = nts

# ── LinUCB ──
print("Training LinUCB (online on train)...")
linucb = LinUCBPolicy(feature_dim=input_dim, alpha=1.0)
for i in range(len(X_train_s)):
    x = X_train_s[i]
    linucb.select_action(np.zeros(N_TREATMENTS), x=x)
    linucb.update_model(x, int(meta_s['a_train'][i]), float(meta_s['y_train'][i]))
models['LinUCB'] = linucb

print(f"\nAll {len(models)} models trained")

## Offline Evaluation — Greedy Actions on Test Set

In [ ]:
results_rows = []

for name, model in models.items():
    if name == 'XGBoost':
        actions = model.predict_best_action(X_test_u)
    elif name == 'LinUCB':
        actions = np.array([model.select_action(np.zeros(N_TREATMENTS), x=X_test_s[i]) for i in range(n_test)])
    else:
        # Neural models
        actions = model.select_actions(X_test_s)

    cf = meta_u['cf_test']
    opt = meta_u['opt_test']

    policy_value = cf[np.arange(n_test), actions].mean()
    oracle_value = cf.max(axis=1).mean()
    regret = oracle_value - policy_value
    accuracy = (actions == opt).mean()

    # Per-treatment accuracy
    per_treatment_acc = {}
    for k in range(N_TREATMENTS):
        mask = opt == k
        if mask.sum() > 0:
            per_treatment_acc[IDX_TO_TREATMENT[k]] = round((actions[mask] == k).mean(), 4)

    results_rows.append({
        "model": name,
        "policy_value": round(policy_value, 4),
        "oracle_value": round(oracle_value, 4),
        "regret": round(regret, 4),
        "accuracy": round(accuracy, 4),
        "actions": actions,
        **{f"acc_{t}": per_treatment_acc.get(t, 0) for t in TREATMENTS},
    })

# Add baselines
random_value = cf.mean()
results_rows.append({
    "model": "Random",
    "policy_value": round(random_value, 4),
    "oracle_value": round(oracle_value, 4),
    "regret": round(oracle_value - random_value, 4),
    "accuracy": round(1.0 / N_TREATMENTS, 4),
    "actions": None,
    **{f"acc_{t}": round(1.0 / N_TREATMENTS, 4) for t in TREATMENTS},
})

results_rows.append({
    "model": "Oracle",
    "policy_value": round(oracle_value, 4),
    "oracle_value": round(oracle_value, 4),
    "regret": 0.0,
    "accuracy": 1.0,
    "actions": None,
    **{f"acc_{t}": 1.0 for t in TREATMENTS},
})

offline_df = pd.DataFrame(results_rows).sort_values("regret")

print("=" * 70)
print("OFFLINE EVALUATION — ALL MODELS")
print("=" * 70)
display_cols = ["model", "policy_value", "regret", "accuracy"]
print(offline_df[display_cols].to_string(index=False))

## Master Comparison Bar Charts

In [ ]:
plot_df = offline_df[offline_df['model'] != 'Oracle'].copy()

fig, axes = plt.subplots(1, 3, figsize=(20, 6))

# Policy Value
ax = axes[0]
colors = AGENT_COLORS[:len(plot_df)]
ax.barh(plot_df['model'], plot_df['policy_value'], color=colors, edgecolor='white')
ax.axvline(oracle_value, color='red', linestyle='--', alpha=0.6, label=f'Oracle={oracle_value:.3f}')
ax.set_xlabel("Policy Value (higher = better)")
ax.set_title("Policy Value")
ax.legend(fontsize=9)
for i, v in enumerate(plot_df['policy_value']):
    ax.text(v + 0.01, i, f"{v:.4f}", va='center', fontsize=9)

# Regret
ax = axes[1]
ax.barh(plot_df['model'], plot_df['regret'], color=colors, edgecolor='white')
ax.set_xlabel("Regret (lower = better)")
ax.set_title("Regret")
for i, v in enumerate(plot_df['regret']):
    ax.text(v + 0.005, i, f"{v:.4f}", va='center', fontsize=9)

# Accuracy
ax = axes[2]
ax.barh(plot_df['model'], plot_df['accuracy'], color=colors, edgecolor='white')
ax.set_xlabel("Accuracy (higher = better)")
ax.set_title("Oracle Match Accuracy")
for i, v in enumerate(plot_df['accuracy']):
    ax.text(v + 0.005, i, f"{v:.4f}", va='center', fontsize=9)

plt.suptitle("Head-to-Head: All Bandit Models", fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig("../results/master_comparison.png", bbox_inches='tight')
plt.show()

## Per-Treatment Accuracy Heatmap

In [ ]:
acc_cols = [f"acc_{t}" for t in TREATMENTS]
model_names = offline_df[~offline_df['model'].isin(['Random', 'Oracle'])]['model'].tolist()

hm_data = offline_df[offline_df['model'].isin(model_names)].set_index('model')[acc_cols]
hm_data.columns = TREATMENTS

fig, ax = plt.subplots(figsize=(12, 7))
sns.heatmap(hm_data, annot=True, fmt='.3f', cmap='YlGn', ax=ax,
            linewidths=0.5, vmin=0, vmax=1,
            cbar_kws={'label': 'Accuracy'})
ax.set_title("Per-Treatment Accuracy by Model")
ax.set_ylabel("Model")
plt.tight_layout()
plt.savefig("../results/per_treatment_accuracy_heatmap.png", bbox_inches='tight')
plt.show()

print("Which treatments are hardest?")
avg_acc = hm_data.mean(axis=0).sort_values()
for t, acc in avg_acc.items():
    print(f"  {t:<12} avg_accuracy={acc:.4f}")

## Confusion Matrices — Top 3 Models

In [ ]:
top3 = offline_df[~offline_df['model'].isin(['Random', 'Oracle'])].head(3)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, (_, row) in zip(axes, top3.iterrows()):
    name = row['model']
    actions = row['actions']
    if actions is None:
        continue
    cm = confusion_matrix(meta_u['opt_test'], actions, labels=list(range(N_TREATMENTS)))
    cm_pct = cm / cm.sum(axis=1, keepdims=True)
    sns.heatmap(cm_pct, annot=True, fmt='.2f', cmap='Blues', ax=ax,
                xticklabels=TREATMENTS, yticklabels=TREATMENTS)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Oracle")
    ax.set_title(f"{name}\n(acc={row['accuracy']:.3f})")

plt.suptitle("Confusion Matrices — Top 3 Models", fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig("../results/top3_confusion_matrices.png", bbox_inches='tight')
plt.show()

## Action Distribution Comparison

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(20, 8))
axes = axes.flatten()

opt_dist = np.bincount(meta_u['opt_test'], minlength=N_TREATMENTS) / n_test

for i, (_, row) in enumerate(offline_df[~offline_df['model'].isin(['Random', 'Oracle'])].iterrows()):
    if i >= 8:
        break
    ax = axes[i]
    name = row['model']
    actions = row['actions']
    if actions is None:
        continue

    model_dist = np.bincount(actions, minlength=N_TREATMENTS) / n_test

    x = np.arange(N_TREATMENTS)
    w = 0.35
    ax.bar(x - w/2, model_dist, w, label='Model', color='#2196F3', edgecolor='white')
    ax.bar(x + w/2, opt_dist, w, label='Oracle', color='#1565C0', edgecolor='white')
    ax.set_xticks(x)
    ax.set_xticklabels(TREATMENTS, fontsize=7, rotation=45)
    ax.set_title(name, fontsize=10)
    ax.set_ylabel("Proportion")
    if i == 0:
        ax.legend(fontsize=8)

# Hide unused axes
for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle("Action Distribution: Each Model vs Oracle", fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

## Online Simulation — All Models Head-to-Head

In [ ]:
print("=" * 70)
print("ONLINE SIMULATION — 20,000 ROUNDS")
print("=" * 70)

seed_everything(99)

# Fresh LinUCB for online
linucb_online = LinUCBPolicy(feature_dim=input_dim, alpha=1.0)

sim_agents = [
    make_random_agent(),
    make_oracle_agent(),
    make_reward_model_agent("XGBoost+Greedy", xgb, GreedyPolicy(), pipe_u),
    make_reward_model_agent("XGBoost+Boltzmann", xgb,
                            BoltzmannPolicy(temperature=0.5, temperature_decay=0.9995), pipe_u),
    make_linucb_agent("LinUCB", linucb_online),
    make_neural_bandit_agent("NeuralUCB", nucb, pipe_s),
    make_neural_bandit_agent("NeuralThompson", nts, pipe_s),
]

sim = OnlineSimulator(n_rounds=20000, pipeline=pipe_s, seed=99, log_interval=5000)
for a in sim_agents:
    sim.add_agent(a)

with timer("Online simulation (20k rounds)"):
    online_results = sim.run()

## Online — Cumulative Regret

In [ ]:
fig = plot_cumulative_regret(
    online_results,
    title="Cumulative Regret — 20,000 Rounds (All Models)",
    save_path="../results/master_cumulative_regret.png",
)
plt.show()

## Online — Learning Curves

In [ ]:
windowed = sim.get_windowed_metrics(window=1000)

fig = plot_regret_and_accuracy(
    windowed,
    save_path="../results/master_learning_curves.png",
)
plt.show()

## Online — Summary Table

In [ ]:
online_summary = sim.get_summary()
print("=" * 70)
print("ONLINE SIMULATION SUMMARY (20K ROUNDS)")
print("=" * 70)
print(online_summary.to_string(index=False))

# Converged performance (last 5000 rounds)
print("\nConverged performance (last 5000 rounds):")
late_rows = []
for name, df_r in online_results.items():
    late = df_r.tail(5000)
    late_rows.append({
        "agent": name,
        "avg_reward": round(late['reward'].mean(), 4),
        "avg_regret": round(late['regret'].mean(), 4),
        "accuracy": round(late['correct'].mean(), 4),
    })
late_df = pd.DataFrame(late_rows).sort_values("avg_regret")
print(late_df.to_string(index=False))

##  Offline vs Online Ranking Comparison

In [ ]:
# Build ranking from both evaluations
offline_rank = offline_df[~offline_df['model'].isin(['Random', 'Oracle'])].copy()
offline_rank['offline_rank'] = range(1, len(offline_rank) + 1)
offline_rank = offline_rank[['model', 'regret', 'offline_rank']].rename(
    columns={'regret': 'offline_regret'})

online_rank = late_df[~late_df['agent'].isin(['random', 'oracle'])].copy()
online_rank['online_rank'] = range(1, len(online_rank) + 1)
online_rank = online_rank[['agent', 'avg_regret', 'online_rank']].rename(
    columns={'agent': 'model', 'avg_regret': 'online_regret'})

# Merge (approximate name matching)
name_map = {
    'XGBoost': 'XGBoost+Greedy',
    'NeuralUCB': 'NeuralUCB',
    'NeuralThompson': 'NeuralThompson',
    'LinUCB': 'LinUCB',
}

rank_rows = []
for off_name, on_name in name_map.items():
    off_row = offline_rank[offline_rank['model'] == off_name]
    on_row = online_rank[online_rank['model'] == on_name]
    if not off_row.empty and not on_row.empty:
        rank_rows.append({
            "model": off_name,
            "offline_regret": off_row.iloc[0]['offline_regret'],
            "offline_rank": off_row.iloc[0]['offline_rank'],
            "online_regret": on_row.iloc[0]['online_regret'],
            "online_rank": on_row.iloc[0]['online_rank'],
        })

rank_df = pd.DataFrame(rank_rows)
print("=" * 70)
print("OFFLINE vs ONLINE RANKING")
print("=" * 70)
print(rank_df.to_string(index=False))

fig, ax = plt.subplots(figsize=(10, 6))
for _, row in rank_df.iterrows():
    ax.plot([0, 1], [row['offline_rank'], row['online_rank']], 'o-', linewidth=2, markersize=10)
    ax.text(-0.05, row['offline_rank'], row['model'], ha='right', va='center', fontsize=10)
    ax.text(1.05, row['online_rank'], row['model'], ha='left', va='center', fontsize=10)

ax.set_xticks([0, 1])
ax.set_xticklabels(["Offline Rank", "Online Rank"], fontsize=12)
ax.set_ylabel("Rank (1 = best)")
ax.set_title("Do Offline and Online Rankings Agree?")
ax.invert_yaxis()
ax.set_xlim(-0.3, 1.3)
plt.tight_layout()
plt.savefig("../results/offline_vs_online_ranking.png", bbox_inches='tight')
plt.show()

## Subgroup Breakdown — All Models

In [ ]:
test_df = df.iloc[meta_u['test_idx']].copy()

# Define subgroups
subgroups = {
    "BMI<30": test_df['bmi'] < 30,
    "BMI≥35": test_df['bmi'] >= 35,
    "HbA1c>10": test_df['hba1c_baseline'] > 10,
    "HbA1c<8": test_df['hba1c_baseline'] < 8,
    "Age>65": test_df['age'] > 65,
    "Age<50": test_df['age'] < 50,
    "CVD=1": test_df['cvd'] == 1,
    "CKD=1": test_df['ckd'] == 1,
}

sub_rows = []
for _, row in offline_df.iterrows():
    name = row['model']
    actions = row['actions']
    if actions is None:
        continue
    for sg_name, sg_mask in subgroups.items():
        mask = sg_mask.values
        if mask.sum() == 0:
            continue
        cf_sub = meta_u['cf_test'][mask]
        act_sub = actions[mask]
        opt_sub = meta_u['opt_test'][mask]
        value = cf_sub[np.arange(mask.sum()), act_sub].mean()
        acc = (act_sub == opt_sub).mean()
        sub_rows.append({
            "model": name,
            "subgroup": sg_name,
            "n": mask.sum(),
            "policy_value": round(value, 4),
            "accuracy": round(acc, 4),
        })

sub_df = pd.DataFrame(sub_rows)

# Pivot for heatmap
acc_pivot = sub_df.pivot(index='model', columns='subgroup', values='accuracy')

fig, ax = plt.subplots(figsize=(14, 7))
sns.heatmap(acc_pivot, annot=True, fmt='.3f', cmap='YlGn', ax=ax,
            linewidths=0.5, vmin=0, vmax=1)
ax.set_title("Model Accuracy Across Patient Subgroups")
ax.set_ylabel("Model")
plt.tight_layout()
plt.savefig("../results/subgroup_accuracy_heatmap.png", bbox_inches='tight')
plt.show()

# Which model is most consistent?
consistency = acc_pivot.std(axis=1).sort_values()
print("Model consistency (std of accuracy across subgroups — lower = more consistent):")
for model, std in consistency.items():
    mean_acc = acc_pivot.loc[model].mean()
    print(f"  {model:<20} mean_acc={mean_acc:.4f}  std={std:.4f}")

## Relative Efficiency Chart

In [ ]:
# How close is each model to the oracle? (percentage)
eff_df = offline_df[~offline_df['model'].isin(['Random', 'Oracle'])].copy()
eff_df['efficiency'] = (eff_df['policy_value'] / oracle_value * 100).round(1)

fig, ax = plt.subplots(figsize=(10, 6))

colors_eff = []
for v in eff_df['efficiency']:
    if v >= 95:
        colors_eff.append('#4CAF50')
    elif v >= 90:
        colors_eff.append('#FF9800')
    else:
        colors_eff.append('#F44336')

ax.barh(eff_df['model'], eff_df['efficiency'], color=colors_eff, edgecolor='white')
ax.axvline(100, color='red', linestyle='--', alpha=0.5, label='Oracle (100%)')
ax.set_xlabel("Relative Efficiency (%)")
ax.set_title("How Close to Oracle? (Policy Value / Oracle Value × 100)")
ax.set_xlim(left=max(0, eff_df['efficiency'].min() - 5))
for i, v in enumerate(eff_df['efficiency']):
    ax.text(v + 0.3, i, f"{v:.1f}%", va='center', fontsize=10, fontweight='bold')
ax.legend()

plt.tight_layout()
plt.savefig("../results/relative_efficiency.png", bbox_inches='tight')
plt.show()

## Strengths & Weaknesses Summary Table

In [ ]:
print("=" * 70)
print("MODEL STRENGTHS & WEAKNESSES")
print("=" * 70)

analysis = []
for _, row in offline_df[~offline_df['model'].isin(['Random', 'Oracle'])].iterrows():
    name = row['model']
    best_treatment = max(TREATMENTS, key=lambda t: row.get(f'acc_{t}', 0))
    worst_treatment = min(TREATMENTS, key=lambda t: row.get(f'acc_{t}', 0))

    analysis.append({
        "model": name,
        "regret": row['regret'],
        "accuracy": row['accuracy'],
        "best_at": f"{best_treatment} ({row.get(f'acc_{best_treatment}', 0):.3f})",
        "worst_at": f"{worst_treatment} ({row.get(f'acc_{worst_treatment}', 0):.3f})",
    })

analysis_df = pd.DataFrame(analysis)
print(analysis_df.to_string(index=False))

## Final Recommendation

In [ ]:
# Determine winner
best_offline = offline_df[~offline_df['model'].isin(['Random', 'Oracle'])].iloc[0]
best_online = late_df[~late_df['agent'].isin(['random', 'oracle'])].iloc[0]

print("=" * 70)
print("FINAL MODEL RECOMMENDATION")
print("=" * 70)

print(f"\n  OFFLINE WINNER:  {best_offline['model']}")
print(f"    Policy value:  {best_offline['policy_value']:.4f}")
print(f"    Regret:        {best_offline['regret']:.4f}")
print(f"    Accuracy:      {best_offline['accuracy']:.4f}")

print(f"\n  ONLINE WINNER:   {best_online['agent']}")
print(f"    Avg regret:    {best_online['avg_regret']:.4f}")
print(f"    Accuracy:      {best_online['accuracy']:.4f}")

print(f"\n  Oracle value:    {oracle_value:.4f}")
print(f"  Random value:    {random_value:.4f}")
print(f"  Improvement over random: {(best_offline['policy_value'] - random_value):.4f}")
print(f"  Efficiency:      {best_offline['policy_value']/oracle_value*100:.1f}% of oracle")

## Save All Final Results

In [ ]:
save_results({
    "offline_comparison": offline_df.drop(columns=['actions']).to_dict(orient='records'),
    "online_summary": online_summary.to_dict(orient='records'),
    "online_converged": late_df.to_dict(orient='records'),
    "ranking_comparison": rank_df.to_dict(orient='records'),
    "subgroup_analysis": sub_df.to_dict(orient='records'),
    "model_analysis": analysis_df.to_dict(orient='records'),
    "recommendation": {
        "offline_winner": best_offline['model'],
        "online_winner": best_online['agent'],
        "oracle_value": oracle_value,
        "best_policy_value": best_offline['policy_value'],
        "best_accuracy": best_offline['accuracy'],
    },
}, path="../results/final_comparison_results.json")

print("All results saved to ../results/final_comparison_results.json")

## Cell 19: Summary

In [ ]:
print()
print("=" * 70)
print("PROJECT COMPLETE — DIABETES CONTEXTUAL BANDITS")
print("=" * 70)

print("""
Models evaluated:
  ✅ XGBoost Ensemble (reward model + greedy/softmax policy)
  ✅ NeuralGreedy, NeuralEpsilon, NeuralUCB, NeuralThompson
  ✅ LinUCB (fully online)

Evaluation methods:
  ✅ Counterfactual ground truth (synthetic oracle)
  ✅ OPE: IPS, SNIPS, DM, DR
  ✅ Online simulation (10K, 20K, 50K rounds)
  ✅ Statistical significance tests
  ✅ Subgroup analysis (BMI, HbA1c, Age, CVD, CKD)

Key deliverables:
  📊 results/master_comparison.png
  📊 results/master_cumulative_regret.png
  📊 results/master_learning_curves.png
  📊 results/per_treatment_accuracy_heatmap.png
  📊 results/subgroup_accuracy_heatmap.png
  📊 results/relative_efficiency.png
  📊 results/offline_vs_online_ranking.png
  📊 results/final_comparison_results.json
""")
print("=" * 70)